# Bank of sKF and fKF filters (Section 4, Algorithms 1 and 2)

This notebook implements the two banks of Section 4 of the draft (`main.tex`, Overleaf commit `fc54c65`,
lines 1162-1466), with the Laplacian and the Gaussian likelihoods. This is item 1 of the draft's own
open-task list (line 1644):

- **Algorithm 1**, bank of sKFs: one sKF per value $\varepsilon^{(j)}$ on a log grid, mixed at every
  step through the transition matrix (eps.chain), eqs. (bank.mixing.prob)-(bank.output).
- **Algorithm 2**, bank of fKFs: one fKF per fixed predicted variance $v^{(j)}$, $\sigma_t^2 = v^{(j)}\|\boldsymbol{x}_t\|^2$,
  the same mixing of the locations, no variance to carry.

Then it compares the two banks with the six single filters of Section 3 (minorized, exact, matched; sKF and fKF),
each at the value that gives a $-20$ dB floor in our scenario.

**Choices made with Ramiro before writing code (2026-10-01)**

| doubt | choice |
|---|---|
| brute-force check of the weights | enumerate every $\varepsilon$ sequence on a small problem (one filter per sequence, no merging) and compare its $\Pr(\varepsilon_t = j \mid y_{1:t})$ with $\mu_{t,j}$ |
| runs besides our scenario | white input; Leszek's drifting system as an anchor; the single exact filter tuned to the bank's own floor |
| initial $v_0$ of the sKF bank | 2 in our scenario (our `VAR_THETA_0`), $1/M$ in Leszek's anchor (his value) |
| Gaussian likelihood | $v_\eta$ = the true noise variance (as Leszek's `driftcmp.py`); in all checks, and in one table of its own, not in the 8-filter comparison |

**Scenario.** Notebook 09's, as in `clipped-matched.ipynb` section 2: AR($-0.9$) input, room response
($M = 128$, $T_{60} = 0.2$ s, 8 kHz), generalized Gaussian noise $\beta^* = 0.2$, 5 dB, $b_\eta = \mathrm{E}|\eta_t|$,
$R = 20$, $v_0 = 2$, updates once the window is full ($t \ge M$). Change test of notebook 09: $2N$ steps with
$\boldsymbol{h}_o \to -\boldsymbol{h}_o$ at $t = N$.

**Grids.** sKF bank: $\varepsilon \in \{10^{-9}, 10^{-8}, \dots, 10^{-4}\}$ ($K = 6$, the draft's remark, line 1614).
fKF bank: $v \in$ geomspace$(10^{-6}, 10^{-2}, 9)$ ($K = 9$, the grid of Leszek's `fkfbank*.py`; it brackets the
tuned $2.985 \times 10^{-4}$). $q = 10^{-4}$ for both.

## 1. Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import quad
from scipy.signal import lfilter
from scipy.special import gamma as G
from scipy.special import gammaln, log_ndtr, logsumexp
from scipy.stats import gennorm, norm
import rir_generator as rir

%config InlineBackend.figure_format = 'svg'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Notebook 09's constants and signals, from `leszek-setup.ipynb` section 4 (commit `adb148b`), where the AR
coefficient is a parameter so the same lines give the white-input run. With `ar_a = -0.9` they are notebook
09's `generate_signals` and `generate_signals_change`, draw for draw (`leszek-setup.ipynb` reproduces notebook
09's numbers with them).

In [ ]:
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run at rest; the change run has 2N
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
SNR_DB = 5.0                # nominal signal-to-noise ratio
VAR_THETA_0 = 2.0           # initial prior variance on each weight (ours; Leszek's is 1/M)
FS = 8000                   # sampling rate [Hz]
TARGET_DB = -20.0           # floor the single filters are tuned to
DRIFT_DB = 0.5              # settling rule of notebook 09

# === copied from leszek-setup.ipynb section 4 (commit adb148b), NOT edited ===
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM, reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))


def noise_of(ar_a):
    # Noise scale and b_eta = E|eta| at 5 dB for this input: notebook 09's lines, AR as a parameter.
    P_signal = ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho
    var_eta = P_signal/10**(SNR_DB/10)
    scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    return scale_gg, scale_gg*np.exp(gammaln(2/BETA) - gammaln(1/BETA))


def signals(seed, ar_a, n, change_at=None):
    # generate_signals / generate_signals_change of notebook 09 (commit 6e6dabc), with the AR
    # coefficient, the length and the instant of the flip as parameters. Same draws, same order.
    scale_gg, _ = noise_of(ar_a)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(n + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    y = np.convolve(ho, x)[:n]
    if change_at is not None:
        y[change_at:] = -y[change_at:]
    eta = gennorm.rvs(BETA, scale=scale_gg, size=n, random_state=rng)
    return x, y + eta
# === end of the copied block ===


def noise_variance(ar_a):
    # The true noise variance, for the Gaussian likelihood: the variance of the generalized Gaussian.
    scale_gg, _ = noise_of(ar_a)
    return scale_gg**2*np.exp(gammaln(3/BETA) - gammaln(1/BETA))


def stack(sigs):
    return np.array([s[0] for s in sigs]), np.array([s[1] for s in sigs])


for ar_a in (-0.9, 0.0):
    scale_gg, b_eta = noise_of(ar_a)
    print(f"AR {ar_a:>4}: var_eta = {noise_variance(ar_a):.4e}, noise scale = {scale_gg:.4e}, "
          f"b_eta = E|eta| = {b_eta:.4e}")

## 3. The filters of Section 3

Copied read-only, nothing edited:

| piece | source |
|---|---|
| `log_mills`, `chi_laplacian` (eqs. Lap.chi, NL.density) | notebook 07 (commit `9406077`), via `clipped-matched.ipynb` (`adb148b`) |
| `chi_gaussian` (eq. chi.Gaussian) | `vkf-kf.ipynb` (commit `2724d95`) |
| `chi_minorized` (eq. chi.minorized) | `fkf.ipynb` (commit `058a9f9`), via `clipped-matched.ipynb` |
| `gg_scale`, `density_params`, `vectorised_gg` (matched filter, quadrature) | `clipped-matched.ipynb` (`adb148b`), checked there against notebook 08's `chi_quadrature` |
| `sKF_batch`, `fKF_batch` | `clipped-matched.ipynb` (`adb148b`) |

In [ ]:
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/07_skf_conjunto.ipynb, commit 9406077,
#         via clipped-matched.ipynb (commit adb148b)
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


# Copied from vkf-kf.ipynb, commit 2724d95.
def chi_gaussian(e, sigma, v_eta):
    """Eq. (38), the Gaussian instance. Used only for the Kalman cross-check of section 4."""
    frac = sigma**2/(sigma**2 + v_eta)
    return frac*e, frac


# Copied from clipped-matched.ipynb, commit adb148b (chi_minorized from fkf.ipynb, commit 058a9f9).
def chi_minorized(e, sigma, b_eta):
    """Eq. (45) of the draft (chi.minorized): the minorized correction tau e/(tau + |e|), and the
    ratio chi/e that replaces chi' in the variance update (Table 2). The fKF uses the first only."""
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta
    ratio = tau/(tau + np.abs(e))                      # chi_min / e, in (0, 1]
    return ratio*e, ratio


# === copied from clipped-matched.ipynb, commit adb148b, NOT edited ===
def gg_scale(beta, var_eta):
    """alpha(beta): the generalized Gaussian of shape beta closest in KL to the simulated noise."""
    alpha_true = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))                 # alpha_*
    mean_abs_pow = alpha_true**beta*np.exp(gammaln((beta + 1)/BETA) - gammaln(1/BETA))    # E|eta|^beta
    return (beta*mean_abs_pow)**(1/beta)                                                   # alpha(beta)


L_WINDOW = 10               # the prior N(s; 0, sigma^2) is below e^{-50} beyond L_WINDOW sigma


def density_params(density, shape, scale, n_nodes):
    """Everything chi_quadrature needs about one density: shape, scale and the nodes."""
    nodes, weights = np.polynomial.legendre.leggauss(n_nodes)   # on [-1, 1], computed once
    nodes = (nodes + 1)/2                              # moved to [0, 1]: a piece [a, b] takes
    weights = weights/2                                # a + (b - a) node, weight (b - a) weight
    if density == "gg":
        return {"beta": shape, "alpha": scale, "nodes": nodes, "log_weights": np.log(weights)}
    return {"nu": shape, "scale": scale, "nodes": nodes, "log_weights": np.log(weights)}


def vectorised_gg(params):
    """chi_quadrature of notebook 08 for the generalized Gaussian with beta < 1, over arrays of
    errors at once: the same three pieces, the same change of variable (piece_in_z), the same nodes
    and weights. New here; checked against chi_quadrature below. b_eta is not used."""
    alpha, beta = params["alpha"], params["beta"]
    nodes, log_weights = params["nodes"], params["log_weights"]

    def chi_fn(e, sigma, b_eta=None):
        e = np.asarray(e, dtype=float)
        sigma = np.broadcast_to(np.asarray(sigma, dtype=float), e.shape)
        # cuts at -L sigma, 0, e_t and +L sigma, sorted: three pieces per error
        cuts = np.sort(np.stack([-L_WINDOW*sigma, np.zeros_like(e), e, L_WINDOW*sigma], axis=-1), axis=-1)
        s_a, s_b = cuts[..., :3], cuts[..., 1:]
        e_col = e[..., None]
        side = np.where(s_a >= e_col, 1.0, -1.0)      # the piece lies above or below e_t
        z_a = (np.abs(e_col - s_a)/alpha)**beta        # z = (|e_t - s|/alpha)^beta at both ends
        z_b = (np.abs(e_col - s_b)/alpha)**beta
        z_low = np.minimum(z_a, z_b)
        length = np.maximum(z_a, z_b) - z_low
        with np.errstate(divide="ignore", invalid="ignore"):   # empty pieces, masked just below
            z = z_low[..., None] + length[..., None]*nodes
            log_z = np.log(z)
            s = e_col[..., None] + side[..., None]*alpha*np.exp(log_z/beta)
            log_q = (log_weights + np.log(length*alpha/beta)[..., None] + (1/beta - 1)*log_z - z
                     - s*s/(2*sigma[..., None, None]**2))     # times the prior N(s; 0, sigma^2)
        full = (s_b > s_a)[..., None]                  # e_t on a cut: that piece is empty
        log_q = np.where(full, log_q, -np.inf).reshape(e.shape + (-1,))
        s = np.where(full, s, 0.0).reshape(e.shape + (-1,))
        q = np.exp(log_q - log_q.max(axis=-1, keepdims=True))
        total = q.sum(axis=-1)
        mean = (q*s).sum(axis=-1)/total                # E[s_t | y_1:t]
        var = (q*(s - mean[..., None])**2).sum(axis=-1)/total   # Var[s_t | y_1:t]
        return mean, 1 - var/sigma**2                  # chi and chi', eq. (30)
    return chi_fn


# sKF_batch and fKF_batch: copied from fkf.ipynb, commit 058a9f9 (sKF_batch from vkf-kf.ipynb,
# commit 2724d95), with options added and nothing else changed. Defaults reproduce the originals.
#   flip_at: misalignment against -h from that step on (fkf.ipynb had it for the fKF only)
#   clip:    chi' -> max(chi', 0) in the variance update, eq. (36); the mean update is untouched
#   record:  also return chi'_t and v_t at every step
#   v0:      initial v (VAR_THETA_0 = 2 in every notebook of ours; 1/M in Leszek's ggbench.py)
#   wait:    first step that updates (M: wait for a full window, as every notebook of ours)
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn, flip_at=None, clip=False, record=False,
              v0=VAR_THETA_0, wait=None):
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(v0))
    eps = np.asarray(epsilon, dtype=float)
    mis = np.empty((B, n))
    if record:
        slope_hist, v_hist = np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < first:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - (np.maximum(slope, 0.0) if clip else slope)/L)
        if record:
            slope_hist[:, t], v_hist[:, t] = slope, v
    if record:
        return mis, slope_hist, v_hist
    return mis


def fKF_batch(X, D, h, b_eta, v, chi_fn, flip_at=None, wait=None):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < first:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        chi, _ = chi_fn(e, np.sqrt(v*power), b_eta)
        w = w + x_t*(chi/power)[:, None]
    return mis
# === end of the copied block ===


def chi_matched(ar_a):
    # The matched correction of clipped-matched.ipynb: the true shape and scale, 100 nodes per piece.
    return vectorised_gg(density_params("gg", BETA, gg_scale(BETA, noise_variance(ar_a)), 100))


print("filters ready")

## 4. The bank

New code. One function per step, `bank_step`, which is lines 4-12 of Algorithm 1 (`family` sKF) or lines 4-8
of Algorithm 2 (fKF, `V = None`), and one loop, `bank_batch`, with the conventions of `sKF_batch`: the window
slides in one sample at a time, nothing is updated before the window is full ($t < M$; the bank then does
nothing at all, the weights stay at $1/K$), and the misalignment at step $t$ is that of the output before
the update.

| line of the algorithm | here |
|---|---|
| $p_{ij}$, eq. (eps.chain) | `transition_matrix`: $q$ to each neighbour, $1 - q\,n_i$ to stay |
| $c_{t,j} = \sum_i p_{ij}\mu_{t-1,i}$, $\mu_{t-1,i\mid j} = p_{ij}\mu_{t-1,i}/c_{t,j}$ | `log_c`, `mix` (in logs: `logsumexp` over $i$) |
| $\bar{\boldsymbol w}_{t,j}$, $\bar v_{t,j}$ with the spread $\lVert\boldsymbol w_{t-1,i} - \bar{\boldsymbol w}_{t,j}\rVert^2/M$, eq. (bank.mixing) | `W_bar`, `V_bar` (sKF only) |
| $\tilde v$, $\sigma^2$, $e$, $\chi_t$, $\chi_t'$, $\boldsymbol w_{t,j}$, $v_{t,j}$, eq. (bank.update) | the sKF step of `sKF_batch`, per filter; for the fKF, $\sigma^2 = v^{(j)}\lVert\boldsymbol x_t\rVert^2$ |
| $\log p_{t,j}(e)$, eq. (bank.weights) | `log_density_laplacian`: $\log\phi(u) + \log(\mathcal R(k-u) + \mathcal R(k+u)) - \log 2b_\eta$, as written; `log_density_gaussian`: $\mathcal N(0, \sigma^2 + v_\eta)$ |
| $\ell_j = \log c_{t,j} + \log p_{t,j}(e)$, $\mu_{t,j} \propto e^{\ell_j - \max\ell}$ | `log_mu = ell - logsumexp(ell)`: the same numbers, kept as logs |
| $\hat{\boldsymbol w}_t = \sum_j \mu_{t,j}\boldsymbol w_{t,j}$, eq. (bank.output) | `w_hat` |

The weights are carried as $\log\mu$ and $c$ is formed as $\log c$, so no weight can underflow to 0 and no
$c_{t,j}$ can be 0. Leszek's scripts carry $\mu$ itself and clamp it from below ($10^{-280}$ in `driftbench.py`,
$10^{-200}$ in `driftcmp.py` and `fkfbank*.py`); the draft has no clamp. `clamp` reproduces his, for the anchor
of section 6 only.

In [ ]:
LOG_2PI = np.log(2*np.pi)


def transition_matrix(K, q):
    """p_ij of eq. (eps.chain): q to each neighbour on the grid, 1 - q n_i to stay."""
    P = np.zeros((K, K))
    for i in range(K):
        neighbours = [j for j in (i - 1, i + 1) if 0 <= j < K]   # n_i = 1 at the ends, 2 inside
        P[i, neighbours] = q
        P[i, i] = 1 - q*len(neighbours)
    return P


def log_density_laplacian(e, sigma, b_eta):
    """log p_t(e) of eq. (bank.weights), from eq. (NL.density): log phi(u) + log(R(k-u) + R(k+u)) - log 2b."""
    u = e/sigma                                        # u = e / sigma
    k_t = sigma/b_eta                                  # k = sigma / b_eta
    return (-0.5*u**2 - 0.5*LOG_2PI                    # log phi(u)
            + np.logaddexp(log_mills(k_t - u), log_mills(k_t + u))   # log(R(k - u) + R(k + u))
            - np.log(2*b_eta))


def log_density_gaussian(e, sigma, v_eta):
    """log p_t(e) for Gaussian noise: N(0, sigma^2 + v_eta), eq. (bank.weights)."""
    s2 = sigma**2 + v_eta
    return -0.5*(LOG_2PI + np.log(s2)) - e**2/(2*s2)


# likelihood -> (chi, log density, the third argument both take: b_eta or v_eta)
def likelihood(name, ar_a):
    if name == "Laplacian":
        return chi_laplacian, log_density_laplacian, noise_of(ar_a)[1]
    return chi_gaussian, log_density_gaussian, noise_variance(ar_a)


def bank_step(W, V, log_mu, x_t, d_t, power, grid, log_P, chi_fn, log_density, scale):
    """One step of Algorithm 1 (V given) or Algorithm 2 (V = None) for B banks side by side.
    W (K, B, L) locations, V (K, B) variances, log_mu (K, B) log weights, x_t (B, L), d_t and power (B,).
    Returns the new W, V, log_mu, and log c, log p and u = e/sigma of this step (K, B)."""
    L = W.shape[-1]
    # mixing, eqs. (bank.mixing.prob) and (bank.mixing)
    log_joint = log_P[:, :, None] + log_mu[:, None, :]   # log p_ij mu_{t-1,i}, (i, j, B)
    log_c = logsumexp(log_joint, axis=0)                 # log c_{t,j}
    mix = np.exp(log_joint - log_c[None])                # mu_{t-1,i|j}
    W_bar = np.einsum("ijb,ibm->jbm", mix, W)            # w_bar_{t,j}
    if V is not None:
        diff = W[:, None] - W_bar[None]                  # w_{t-1,i} - w_bar_{t,j}, (i, j, B, L)
        spread = np.einsum("ijbm,ijbm->ijb", diff, diff)/L
        V_bar = np.einsum("ijb,ijb->jb", mix, V[:, None] + spread)   # v_bar_{t,j}
        v_tilde = V_bar + grid[:, None]                  # v_tilde = v_bar + eps^(j)
    else:
        v_tilde = np.broadcast_to(grid[:, None], log_mu.shape)       # fKF: the fixed v^(j)
    # one sKF / fKF step per filter, eq. (bank.update)
    sigma = np.sqrt(v_tilde*power)                       # sigma_{t,j}
    e = d_t - np.einsum("jbm,bm->jb", W_bar, x_t)        # e_{t,j}, each filter's own error
    chi, slope = chi_fn(e, sigma, scale)
    W = W_bar + x_t*(chi/power)[..., None]
    if V is not None:
        V = v_tilde*(1 - slope/L)
    # weights, eq. (bank.weights), in logs
    log_p = log_density(e, sigma, scale)
    ell = log_c + log_p
    log_mu = ell - logsumexp(ell, axis=0)
    return W, V, log_mu, log_c, log_p, e/sigma


def bank_batch(X, D, h, grid, q, family, chi_fn, log_density, scale, v0=VAR_THETA_0, flip_at=None,
               wait=None, record=False, clamp=None):
    """B banks side by side, one per row of X and D. Returns the misalignment of the output (B, n),
    the weights averaged over the rows (K, n) and the largest |u| = |e|/sigma met. record=True adds,
    per filter and step, log mu, log c, log p and the misalignment of each filter (K, B, n), and the
    output after each step (B, n, L)."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    grid = np.asarray(grid, dtype=float)
    K = len(grid)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))          # log 0 = -inf where p_ij = 0
    W = np.zeros((K, B, L))
    V = np.full((K, B), float(v0)) if family == "sKF" else None
    log_mu = np.full((K, B), -np.log(K))                 # mu_0 = 1/K
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    mis = np.empty((B, n))
    mu_mean = np.empty((K, n))
    u_max = 0.0
    if record:
        rec = {key: np.full((K, B, n), np.nan) for key in ("log_mu", "log_c", "log_p", "mis_j")}
        rec["w_hat"] = np.full((B, n, L), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        target = h if flip_at is None or t < flip_at else -h
        dw = w_hat - target
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if record:
            dW = W - target
            rec["mis_j"][:, :, t] = np.einsum("jbm,jbm->jb", dW, dW)
        if t >= first:
            power = np.einsum("bm,bm->b", x_t, x_t)
            W, V, log_mu, log_c, log_p, u = bank_step(W, V, log_mu, x_t, D[:, t], power, grid, log_P,
                                                      chi_fn, log_density, scale)
            if clamp is not None:                        # Leszek's mu = max(mu, clamp), renormalized
                log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
                log_mu = log_mu - logsumexp(log_mu, axis=0)
            w_hat = np.einsum("jb,jbm->bm", np.exp(log_mu), W)          # eq. (bank.output)
            u_max = max(u_max, float(np.abs(u).max()))
            if record:
                rec["log_c"][:, :, t], rec["log_p"][:, :, t], rec["w_hat"][:, t] = log_c, log_p, w_hat
        mu_mean[:, t] = np.exp(log_mu).mean(axis=1)
        if record:
            rec["log_mu"][:, :, t] = log_mu
    out = {"mis": mis, "mu": mu_mean, "u_max": u_max}
    if record:
        out.update(rec)
    return out


EPS_BANK = np.geomspace(1e-9, 1e-4, 6)     # the draft's remark, line 1614
V_BANK = np.geomspace(1e-6, 1e-2, 9)       # Leszek's fkfbank*.py
Q_BANK = 1e-4
print("eps grid:", " ".join(f"{g:.0e}" for g in EPS_BANK))
print("v grid:  ", " ".join(f"{g:.1e}" for g in V_BANK))

## 5. Checks

Every check runs with both likelihoods (check 4 of the brief).

### 5.1 The log predictive density

Eq. (bank.weights) for the Laplacian against a direct quadrature of the convolution (predictive.density),
$\int \mathcal N(s; 0, \sigma^2)\,\frac{1}{2b_\eta}e^{-|e - s|/b_\eta}\,ds$; its integral over $e$; and,
for large $|u|$, against the same density written without the cancellation between $\log\phi(u) \approx -u^2/2$
and $\log\mathcal R(k - u) \approx +(k-u)^2/2$:
$\log p = k^2/2 + \log\big(e^{-ku}\Phi(u - k) + e^{ku}\Phi(-u - k)\big) - \log 2b_\eta$.

In [ ]:
b_check = noise_of(-0.9)[1]


def log_density_quadrature(e, s, b):
    # log of the convolution integral, with the integrand scaled by its largest value
    log_f = lambda z: -z*z/(2*s*s) - abs(e - z)/b
    top = max(log_f(0.0), log_f(e), log_f(e - np.sign(e)*min(abs(e), s*s/b)))
    lo, hi = min(-12*s, e - 60*b), max(12*s, e + 60*b)
    cuts = sorted({lo, -12*s, 0.0, 12*s, e - 40*b, e, e + 40*b, hi})
    cuts = [c for c in cuts if lo <= c <= hi]
    total = sum(quad(lambda z: np.exp(log_f(z) - top), a, c, limit=400, epsabs=1e-15, epsrel=1e-12)[0]
                for a, c in zip(cuts[:-1], cuts[1:]))
    return np.log(total) + top - 0.5*LOG_2PI - np.log(s) - np.log(2*b)


def log_density_no_cancellation(e, s, b):
    u, k_t = e/s, s/b
    return k_t**2/2 + np.logaddexp(log_ndtr(u - k_t) - k_t*u, log_ndtr(-u - k_t) + k_t*u) - np.log(2*b)


worst = 0.0
for s in (1e-3, 1e-2, 0.1, 1.0, 10.0):
    for e in (0.0, 1e-3, -0.05, 0.3, -1.0, 5.0, 30.0):
        worst = max(worst, abs(float(log_density_laplacian(np.float64(e), np.float64(s), b_check))
                               - log_density_quadrature(e, s, b_check)))
print(f"Laplacian log density against quadrature, 35 points: max |difference| = {worst:.1e}")
for s in (0.01, 0.3, 5.0):
    total = quad(lambda e: np.exp(log_density_laplacian(np.float64(e), np.float64(s), b_check)),
                 -np.inf, np.inf, limit=500)[0]
    print(f"   integral over e at sigma = {s:<5}: {total:.12f}")

worst_g = 0.0
for s in (1e-3, 0.1, 10.0):
    e = np.linspace(-50, 50, 101)
    worst_g = max(worst_g, np.max(np.abs(log_density_gaussian(e, s, 0.3)
                                         - norm.logpdf(e, scale=np.sqrt(s*s + 0.3)))))
print(f"Gaussian log density against scipy's normal: max |difference| = {worst_g:.1e}")

print("\nthe draft's form against the form without cancellation, sigma = 0.05:")
for u in (1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9):
    e = np.float64(u*0.05)
    gap = float(log_density_laplacian(e, np.float64(0.05), b_check) - log_density_no_cancellation(e, 0.05, b_check))
    print(f"   |u| = {u:.0e}: difference {gap:+.1e}")

### 5.2 Check 1: $K = 1$

With one value the bank must be the single filter: $p_{11} = 1$, the mixing weight is 1, the spread is 0.
Realisations 0-3 of our scenario, 6000 steps, at the tuned exact values.

In [ ]:
X_chk, D_chk = stack([signals(seed, -0.9, 6000) for seed in range(4)])
for name in ("Laplacian", "Gaussian"):
    chi_fn, log_density, scale = likelihood(name, -0.9)
    for family, value in (("sKF", 1.3493e-6), ("fKF", 2.985e-4)):
        ref = (sKF_batch(X_chk, D_chk, ho, scale, np.full(4, value), chi_fn) if family == "sKF"
               else fKF_batch(X_chk, D_chk, ho, scale, np.full(4, value), chi_fn))
        for q in (0.0, Q_BANK):
            out = bank_batch(X_chk, D_chk, ho, [value], q, family, chi_fn, log_density, scale)
            same = np.array_equal(out["mis"], ref)
            gap = np.max(np.abs(out["mis"] - ref)/ref)
            print(f"{name:<10} {family}  q = {q:<7g} bank (K = 1) against the single filter: "
                  f"{'identical' if same else f'max relative difference {gap:.1e}'}, "
                  f"weights all 1: {np.all(out['mu'] == 1.0)}")

### 5.3 Check 2: $q = 0$

With $q = 0$ the mixing weights are $\delta_{ij}$, so filter $j$ of the bank must be the single filter with
$\varepsilon^{(j)}$ (or $v^{(j)}$) run alone, and the weights must be eq. (eps.likelihood): the normalized
exponential of $\log(1/K) + \sum_t \log p_{t,j}(e_{t,j})$. The reference densities come from the single filters,
not from the bank: `single_errors` is `sKF_batch` / `fKF_batch` with $e_t$ and $\sigma_t$ kept, checked equal to them.

In [ ]:
def single_errors(X, D, value, family, chi_fn, scale, v0=VAR_THETA_0):
    # The loop of sKF_batch / fKF_batch, keeping e_t and sigma_t (NaN before the first update).
    B, n = X.shape
    w, x_t, v = np.zeros((B, M)), np.zeros((B, M)), np.full(B, float(v0))
    mis, e_hist, s_hist = np.empty((B, n)), np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        mis[:, t] = np.einsum("bm,bm->b", w - ho, w - ho)
        if t < M:
            continue
        v_tilde = v + value if family == "sKF" else np.full(B, value)
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, scale)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - slope/M)
        e_hist[:, t], s_hist[:, t] = e, sigma
    return mis, e_hist, s_hist


for name in ("Laplacian", "Gaussian"):
    chi_fn, log_density, scale = likelihood(name, -0.9)
    for family, grid in (("sKF", EPS_BANK), ("fKF", V_BANK)):
        out = bank_batch(X_chk, D_chk, ho, grid, 0.0, family, chi_fn, log_density, scale, record=True)
        filter_gap, loop_same, cum = 0.0, True, []
        for j, value in enumerate(grid):
            mis_j, e_j, s_j = single_errors(X_chk, D_chk, value, family, chi_fn, scale)
            ref = (sKF_batch(X_chk, D_chk, ho, scale, np.full(4, value), chi_fn) if family == "sKF"
                   else fKF_batch(X_chk, D_chk, ho, scale, np.full(4, value), chi_fn))
            loop_same &= np.array_equal(mis_j, ref)
            filter_gap = max(filter_gap, np.max(np.abs(out["mis_j"][j] - ref)/ref))
            with np.errstate(invalid="ignore"):            # e_t, sigma_t are NaN before the first update
                log_p_j = log_density(e_j, s_j, scale)
            cum.append(np.cumsum(np.nan_to_num(log_p_j, nan=0.0), axis=1))
        log_mu_ref = np.log(1/len(grid)) + np.array(cum)                       # eq. (eps.likelihood)
        log_mu_ref = log_mu_ref - logsumexp(log_mu_ref, axis=0)
        weight_gap = np.max(np.abs(np.exp(out["log_mu"][:, :, M:]) - np.exp(log_mu_ref[:, :, M:])))
        log_gap = np.max(np.abs(out["log_mu"][:, :, M:] - log_mu_ref[:, :, M:]))
        print(f"{name:<10} {family} K = {len(grid)}: filters against single filters, max relative difference "
              f"{filter_gap:.1e}; weights against eq. (eps.likelihood), max |mu difference| {weight_gap:.1e}, "
              f"max |log mu difference| {log_gap:.1e}   (reference loop = sKF/fKF_batch: {loop_same})")

### 5.4 Check 3: sums, and very large errors

Rows of $p_{ij}$ sum to 1. In a run, at every step: $\sum_j c_{t,j} = 1$, $\sum_j \mu_{t,j} = 1$ and
$\sum_i \mu_{t-1,i\mid j} = 1$ for every $j$. To test the log domain, outliers of $\pm 10^3$, $\pm 10^6$ and $\pm 10^9$
are added to $y_t$ at a few steps, on top of a flip of $\boldsymbol h_o$ at step 3000; every quantity must stay finite.
Each bank runs twice, without and with the outliers.

In [ ]:
worst_row = max(np.max(np.abs(transition_matrix(K, q).sum(axis=1) - 1))
                for K in (1, 2, 3, 6, 9) for q in (0.0, 1e-4, 1e-2, 0.3))
print(f"rows of p_ij, K = 1, 2, 3, 6, 9 and q = 0, 1e-4, 1e-2, 0.3: max |row sum - 1| = {worst_row:.1e}")

X_big, D_big = stack([signals(seed, -0.9, 6000, change_at=3000) for seed in range(4)])
spikes = np.array([700, 1500, 2200, 3300, 4100, 5000])
D_spiked = D_big.copy()
D_spiked[:, spikes] += np.array([1e3, -1e6, 1e9, -1e3, 1e6, -1e9])
print(f"{'':<16}{'outliers':<10}{'sum c - 1':>11}{'sum mu - 1':>12}{'sum_i mu_i|j - 1':>18}{'finite':>8}{'largest |u|':>13}")
for name in ("Laplacian", "Gaussian"):
    chi_fn, log_density, scale = likelihood(name, -0.9)
    for family, grid in (("sKF", EPS_BANK), ("fKF", V_BANK)):
        for label, D_run in (("none", D_big), ("added", D_spiked)):
            out = bank_batch(X_big, D_run, ho, grid, Q_BANK, family, chi_fn, log_density, scale,
                             flip_at=3000, record=True)
            steps = slice(M, None)
            log_mu, log_c = out["log_mu"][:, :, steps], out["log_c"][:, :, steps]
            with np.errstate(divide="ignore"):
                log_P = np.log(transition_matrix(len(grid), Q_BANK))
            prev = out["log_mu"][:, :, M - 1:-1]                   # log mu_{t-1}
            mix_sums = np.exp(log_P[:, :, None, None] + prev[:, None] - log_c[None]).sum(axis=0)
            finite = all(np.isfinite(a).all() for a in (out["mis"], log_mu, log_c, out["log_p"][:, :, steps]))
            print(f"{name + ', ' + family:<16}{label:<10}{np.max(np.abs(np.exp(log_c).sum(axis=0) - 1)):>11.1e}"
                  f"{np.max(np.abs(np.exp(log_mu).sum(axis=0) - 1)):>12.1e}{np.max(np.abs(mix_sums - 1)):>18.1e}"
                  f"{str(finite):>8}{out['u_max']:>13.1e}")
print("   largest |sum - 1| over all steps, filters and realisations")

### 5.5 Check 5: the brute-force mixture over $\varepsilon$ sequences

Without the merge, the posterior after $t$ updates carries one filter per sequence
$\varepsilon_1, \dots, \varepsilon_t$ of positive probability (Section 4.3, "the third"). `enumerate_sequences`
propagates all of them: each one an sKF (or fKF) step with the $\varepsilon$ (or $v$) of its sequence, from its own
state, its weight multiplied by $p_{ij}$ and by its own predictive density. Summing the weights of the sequences that
end at $j$ gives the exact $\Pr(\varepsilon_t = \varepsilon^{(j)} \mid y_{1:t})$ under the per-step Gaussian
projection, with no merge. The bank must match it exactly at the first update (every sequence starts from the same
state, so merging loses nothing), and at every step when $q = 0$ (only the $K$ constant sequences have positive
probability). After the first update, with $q > 0$, the difference is what the merge costs.

Small problem: $M = 4$, $K = 3$, white input, a unit-norm random response that flips sign after the 6th update,
Laplacian noise $b = 0.1$, $v_0 = 0.01$, 12 updates (up to 114 243 sequences).

In [ ]:
def enumerate_sequences(x, d, grid, q, family, chi_fn, log_density, scale, v0):
    """One filter per eps sequence, no merging. Returns Pr(eps_t = j | y_1:t) (n, K), the mixture mean of
    theta (n, L) and the number of sequences at each step."""
    L, K, n = M_SMALL, len(grid), len(x)
    P = transition_matrix(K, q)
    W, V = np.zeros((K, L)), np.full(K, float(v0))
    log_w, last = np.full(K, -np.log(K)), np.arange(K)     # eps_0 = j with probability 1/K
    x_t = np.zeros(L)
    probs, means, count = np.full((n, K), np.nan), np.full((n, L), np.nan), np.zeros(n, dtype=int)
    for t in range(n):
        x_t = np.roll(x_t, 1)
        x_t[0] = x[t]
        if t < L:
            continue
        parent, child = np.nonzero(P[last] > 0)            # every move i -> j of positive probability
        log_w = log_w[parent] + np.log(P[last[parent], child])
        W, V, last = W[parent], V[parent], child
        power = x_t @ x_t
        v_tilde = V + grid[last] if family == "sKF" else grid[last]
        sigma = np.sqrt(v_tilde*power)
        e = d[t] - W @ x_t
        chi, slope = chi_fn(e, sigma, scale)
        W = W + np.outer(chi/power, x_t)
        if family == "sKF":
            V = v_tilde*(1 - slope/L)
        log_w = log_w + log_density(e, sigma, scale)
        log_w = log_w - logsumexp(log_w)
        probs[t] = np.bincount(last, weights=np.exp(log_w), minlength=K)
        means[t] = np.exp(log_w) @ W
        count[t] = len(log_w)
    return probs, means, count


M_SMALL, UPDATES, B_SMALL, V0_SMALL = 4, 12, 0.1, 0.01
rng_small = np.random.default_rng(5)
h_small = rng_small.standard_normal(M_SMALL)
h_small = h_small/np.linalg.norm(h_small)
n_small = M_SMALL + UPDATES
x_small = rng_small.standard_normal(n_small)
y_small = np.convolve(h_small, x_small)[:n_small]
y_small[M_SMALL + 6:] = -y_small[M_SMALL + 6:]
d_small = y_small + rng_small.laplace(0, B_SMALL, n_small)
SMALL_GRID = {"sKF": np.array([1e-4, 1e-3, 1e-2]), "fKF": np.array([1e-3, 1e-2, 1e-1])}
SMALL_LIK = {"Laplacian": (chi_laplacian, log_density_laplacian, B_SMALL),
             "Gaussian": (chi_gaussian, log_density_gaussian, 2*B_SMALL**2)}

brute = {}
print(f"{'':<16}{'q':>7}{'1st update':>12}{'2nd':>10}{'largest':>10}{'output, largest':>18}{'sequences':>11}")
for family, grid in SMALL_GRID.items():
    for name, (chi_fn, log_density, scale) in SMALL_LIK.items():
        for q in (0.0, 1e-4, 1e-2, 0.2):
            probs, means, count = enumerate_sequences(x_small, d_small, grid, q, family, chi_fn, log_density, scale,
                                                      V0_SMALL)
            out = bank_batch(x_small[None], d_small[None], h_small, grid, q, family, chi_fn, log_density, scale,
                             v0=V0_SMALL, record=True)
            mu = np.exp(out["log_mu"][:, 0, M_SMALL:]).T                 # (updates, K)
            gap = np.abs(mu - probs[M_SMALL:]).max(axis=1)
            w_gap = (np.linalg.norm(out["w_hat"][0, M_SMALL:] - means[M_SMALL:], axis=1)
                     / np.linalg.norm(means[M_SMALL:], axis=1))
            brute[(family, name, q)] = gap
            print(f"{family + ', ' + name:<16}{q:>7g}{gap[0]:>12.1e}{gap[1]:>10.1e}{gap.max():>10.1e}"
                  f"{w_gap.max():>18.1e}{count[-1]:>11d}")
print("   1st update / 2nd / largest: max over j of |mu_t,j - Pr(eps_t = j | y_1:t)|, over the 12 updates")
print("   output: largest ||w_hat_t - mixture mean_t|| / ||mixture mean_t||")

## 6. Anchor: Leszek's drifting system

His signals, drawn as `sims/driftbench.py` draws them (Overleaf commit `fc54c65`): white unit-power input,
$R = 40$, 5 dB, $\beta^* = 0.2$, $b_\eta = \mathrm{E}|\eta|$, a random response with a 0.2 s decay; the response
drifts with variance $10^{-8}, 10^{-7}, 10^{-6}, 10^{-8}$ per coefficient and sample in four segments of 6000
samples and changes sign at the three boundaries. His loop is written again around `bank_step`: his regressor
`x[t:t+M][::-1]` is a full window from $t = 0$, so he updates from the first step; $v_0 = 1/M$; his misalignment is
taken after the update and divided by $\lVert\boldsymbol H_t\rVert^2$; the score is the mean over the record, in dB.
A fixed $\varepsilon$ is the bank with $K = 1$ (check 1).

Numbers to reproduce: `sims/README.md` (2026-09-22 rerun of `driftbench.py` and `driftbench_q0.py`) and the
draft's table (line 1623, rows "white", from `driftcmp.py`, one decimal).

In [ ]:
# Written from Leszek's sims/driftbench.py (Overleaf commit fc54c65), AR = 0, line for line.
R_L, SEG_L, EPS_TRUE_L = 40, 6000, [1e-8, 1e-7, 1e-6, 1e-8]
T_L = SEG_L*len(EPS_TRUE_L)
rng0 = np.random.default_rng(3)
x_l = rng0.standard_normal((R_L, T_L + M))       # AR = 0: his AR loop leaves x = u, and multiplies by 1
veta_l = 10**(-SNR_DB/10)                        # E[(x'h)^2] = ||h||^2 = 1
alpha_l = np.sqrt(veta_l*G(1/BETA)/G(3/BETA))    # gg_alpha of his ggq.py
eta_l = np.sign(rng0.standard_normal((R_L, T_L)))*alpha_l*rng0.gamma(1/BETA, 1, (R_L, T_L))**(1/BETA)
b_l = np.sqrt(veta_l)*G(2/BETA)/np.sqrt(G(1/BETA)*G(3/BETA))    # E|eta|
h0_l = rng0.standard_normal(M)*np.exp(-6.9*np.arange(M)/(0.2*8000))
h0_l = h0_l/np.linalg.norm(h0_l)


def system_l():
    # his system(): yields H_t (R, M); the same path in every call
    r = np.random.default_rng(11)
    H = np.tile(h0_l, (R_L, 1))
    for t in range(T_L):
        if t > 0 and t % SEG_L == 0:
            H = -H                                   # abrupt change: sign flip
        H = H + np.sqrt(EPS_TRUE_L[t//SEG_L])*r.standard_normal((R_L, M))
        yield H


def run_leszek(grid, q, family, chi_fn, log_density, scale, clamp=None):
    # his run_bank (driftbench.py, driftcmp.py), through bank_step; run_fixed is K = 1, q = 0
    grid = np.asarray(grid, dtype=float)
    K = len(grid)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))
    W = np.zeros((K, R_L, M))
    V = np.full((K, R_L), 1.0/M) if family == "sKF" else None
    log_mu = np.full((K, R_L), -np.log(K))
    mis = np.empty(T_L)
    for t, H in enumerate(system_l()):
        X = x_l[:, t:t + M][:, ::-1]
        n2 = np.einsum("rm,rm->r", X, X)
        y = np.einsum("rm,rm->r", X, H) + eta_l[:, t]
        W, V, log_mu, *_ = bank_step(W, V, log_mu, X, y, n2, grid, log_P, chi_fn, log_density, scale)
        if clamp is not None:
            log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
            log_mu = log_mu - logsumexp(log_mu, axis=0)
        w = np.einsum("jr,jrm->rm", np.exp(log_mu), W)
        mis[t] = np.mean(np.sum((w - H)**2, axis=1)/np.sum(H**2, axis=1))
    return 10*np.log10(mis.mean())


LIK_L = {"Laplacian": (chi_laplacian, log_density_laplacian, b_l),
         "Gaussian": (chi_gaussian, log_density_gaussian, veta_l)}
start = time.time()
anchor = {}
for name, lik in LIK_L.items():
    fixed = {eps: run_leszek([eps], 0.0, "sKF", *lik) for eps in np.geomspace(1e-9, 1e-4, 11)}
    best = min(fixed, key=fixed.get)
    anchor[(name, "best fixed")] = (best, fixed[best])
    print(f"{name}, fixed eps: " + ", ".join(f"{eps:.1e} {m:.2f}" for eps, m in fixed.items()))

RUNS = [("Laplacian", 1e-4, None), ("Laplacian", 1e-4, 1e-280), ("Laplacian", 1e-3, None),
        ("Laplacian", 0.0, 1e-280), ("Laplacian", 0.0, None), ("Gaussian", 1e-4, None), ("Gaussian", 1e-4, 1e-200)]
for name, q, clamp in RUNS:
    anchor[(name, q, clamp)] = run_leszek(EPS_BANK, q, "sKF", *LIK_L[name], clamp=clamp)

HIS = {("Laplacian", "best fixed"): "-8.98 (eps = 1e-5)", ("Laplacian", 1e-4, 1e-280): "-10.75",
       ("Laplacian", 0.0, 1e-280): "-9.87", ("Laplacian", 1e-3, None): "same as q = 1e-4 (draft)",
       ("Gaussian", "best fixed"): "-7.7 (draft's table)", ("Gaussian", 1e-4, 1e-200): "-8.5 (draft's table)"}
print(f"\n{'run':<44}{'ours [dB]':>10}   Leszek")
for key, value in anchor.items():
    if key[1] == "best fixed":
        label, ours = f"{key[0]}, best fixed eps ({value[0]:.0e})", value[1]
    else:
        label = f"{key[0]}, bank, q = {key[1]:g}, " + ("no clamp" if key[2] is None else f"clamp {key[2]:.0e}")
        ours = value
    print(f"{label:<44}{ours:>10.2f}   {HIS.get(key, '')}")
print(f"[{time.time() - start:.0f} s]")

## 7. Our scenario: the two banks and the six single filters

The single filters at the values tuned to a $-20$ dB floor by notebook 09's protocol, as given in the brief:
sKF from `clipped-matched.ipynb` / notebook 09, fKF from `leszek-setup.ipynb` section 4. They are not re-tuned
here (section 7.3 re-tunes the exact ones as a control). The matched filters take the true scale of the noise
and do not use $b_\eta$. The banks use the Laplacian likelihood, $v_0 = 2$, $q = 10^{-4}$.

For each filter, two runs on notebook 09's signals: $N$ steps at rest, and $2N$ steps with the flip at $N$.

- **At rest**: floor = mean of the last quarter, in dB; *settled*: notebook 09's rule; *own + 3*: first step within
  3 dB of the filter's own floor (notebook 09's "steps"); then the first step below $-17$ dB (the target + 3 dB,
  which is where the tuned filters count as converged) and below $-20$ dB.
- **After the flip**: first step after the change below 0, $-5$, $-10$, $-15$ dB, below its own floor + 3 dB
  (notebook 09's "recovery"), and below $-20$ dB.
- **Mean**: the mean misalignment over the whole $2N$ record, in dB (the draft's metric).

Measurement functions copied from `clipped-matched.ipynb` (`adb148b`), nothing edited.

In [ ]:
# === IGNACIO: steady_state, is_settled and recovery - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc,
#         via clipped-matched.ipynb (commit adb148b)
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


def is_settled(misalignment, drift):
    """Settled within the run: within 3 dB of the floor by half of it, and no longer descending."""
    n = len(misalignment)
    floor, reached = steady_state(misalignment)
    third_quarter = 10*np.log10(misalignment[n//2:3*n//4].mean())
    # reached = 0: the run never left the 3 dB band around its start, so it never converged either
    return bool(0 < reached <= n/2 and third_quarter - floor <= drift)


def recovery(misalignment, floor):
    """Steps from the change until the misalignment is back within 3 dB of the floor at rest;
    -1 if it never gets there before the run ends."""
    db = 10*np.log10(misalignment[N:])
    back = db < floor + 3
    if not back.any():
        return -1
    return int(np.argmax(back))
# === end of the copied block ===


# Copied from clipped-matched.ipynb, commit adb148b.
def time_to(misalignment, level_db):
    """First step after the change below level_db; -1 if never."""
    below = 10*np.log10(misalignment[N:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


LEVELS = [0.0, -5.0, -10.0, -15.0]


def first_below(misalignment, level_db):
    below = 10*np.log10(misalignment) < level_db
    return int(np.argmax(below)) if below.any() else -1


def measure(rest_curve, change_curve):
    floor, reached = steady_state(rest_curve)
    return dict(floor=floor, reached=reached, settled=is_settled(rest_curve, DRIFT_DB),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS], rec=recovery(change_curve, floor),
                back_20=time_to(change_curve, TARGET_DB), mean_db=10*np.log10(change_curve.mean()),
                curve=change_curve)


def show_table(rows):
    print(f"{'':<34}{'at rest':-^46}  {'after the flip':-^50}")
    print(f"{'filter':<22}{'eps or v':>12}{'floor':>8}{'settled':>9}{'own+3':>7}{'-17 dB':>8}{'-20 dB':>8}"
          + "".join(f"{level:>8.0f}" for level in LEVELS) + f"{'own+3':>9}{'-20 dB':>9}{'mean [dB]':>11}")
    for name, r in rows.items():
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{'yes' if r['settled'] else 'NO':>9}{r['reached']:>7d}"
              f"{r['to_17']:>8d}{r['to_20']:>8d}" + "".join(f"{s:>8d}" for s in r["levels"])
              + f"{r['rec']:>9d}{r['back_20']:>9d}{r['mean_db']:>11.2f}")
    print("   steps; -1 = never within the run")


def run_single(family, chi_fn, scale, value, X, D, flip_at=None):
    B = len(X)
    if family == "sKF":
        return sKF_batch(X, D, ho, scale, np.full(B, value), chi_fn, flip_at=flip_at).mean(axis=0)
    return fKF_batch(X, D, ho, scale, np.full(B, value), chi_fn, flip_at=flip_at).mean(axis=0)


def run_bank(family, lik, X, D, flip_at=None):
    out = bank_batch(X, D, ho, EPS_BANK if family == "sKF" else V_BANK, Q_BANK, family, *lik, flip_at=flip_at)
    return out["mis"].mean(axis=0), out["mu"], out["u_max"]


def single_row(family, chi_fn, scale, value, sig):
    (X_r, D_r), (X_c, D_c) = sig
    rest = run_single(family, chi_fn, scale, value, X_r, D_r)
    change = run_single(family, chi_fn, scale, value, X_c, D_c, flip_at=N)
    return dict(value=value, **measure(rest, change))


def bank_row(family, lik, sig):
    (X_r, D_r), (X_c, D_c) = sig
    rest, _, u_rest = run_bank(family, lik, X_r, D_r)
    change, mu, u_change = run_bank(family, lik, X_c, D_c, flip_at=N)
    return dict(value=np.nan, mu=mu, u_max=max(u_rest, u_change), rest_curve=rest, **measure(rest, change))


def scenario_signals(ar_a):
    return (stack([signals(seed, ar_a, N) for seed in range(R)]),
            stack([signals(seed, ar_a, 2*N, change_at=N) for seed in range(R)]))


print("measurement ready")

### 7.1 The eight filters

In [ ]:
start = time.time()
SIG = scenario_signals(-0.9)
_, B_ETA = noise_of(-0.9)
LIK = likelihood("Laplacian", -0.9)
CORRECTION = {"minorized": chi_minorized, "exact": chi_laplacian, "matched": chi_matched(-0.9)}
TUNED = {("sKF", "minorized"): 2.6412e-6, ("sKF", "exact"): 1.3493e-6, ("sKF", "matched"): 2.1851e-6,
         ("fKF", "minorized"): 4.322e-4, ("fKF", "exact"): 2.985e-4, ("fKF", "matched"): 3.658e-4}
# floor, steps at rest and recovery printed by leszek-setup.ipynb section 4 (commit adb148b)
PRINTED = {("sKF", "minorized"): (-19.98, 4357, 7340), ("sKF", "exact"): (-20.01, 4756, 7584),
           ("sKF", "matched"): (-19.99, 3630, 5979), ("fKF", "minorized"): (-19.93, 5246, 7909),
           ("fKF", "exact"): (-19.99, 5461, 8188), ("fKF", "matched"): (-19.98, 4184, 7562)}

rows = {}
for family in ("sKF", "fKF"):
    for method in ("minorized", "exact", "matched"):
        rows[f"{family}, {method}"] = single_row(family, CORRECTION[method], B_ETA, TUNED[(family, method)], SIG)
    rows[f"{family}, bank"] = bank_row(family, LIK, SIG)
    print(f"{family} done [{time.time() - start:.0f} s]")

print("\ncontrol: the single filters against the numbers printed by leszek-setup.ipynb section 4")
for (family, method), (floor, steps, rec) in PRINTED.items():
    r = rows[f"{family}, {method}"]
    print(f"   {family + ', ' + method:<16} floor {r['floor']:.2f} / {floor:.2f}, steps {r['reached']} / {steps}, "
          f"recovery {r['rec']} / {rec}")
print()
show_table(rows)
print(f"largest |u| met by the banks: sKF {rows['sKF, bank']['u_max']:.1e}, fKF {rows['fKF, bank']['u_max']:.1e}")

**Figure 1.** Misalignment across the change, time counted from it, for the eight filters of the table.
Colour = method, line style = family (sKF solid, fKF dashed). Grey: the $-20$ dB target. Black, dash-dot:
the change. Before the change the banks sit at their own levels, not at $-20$ dB. The short upward spikes of the
banks are single realisations; section 8.1 takes the largest one apart. No per-filter recovery markers: with eight
curves and two floors they made the plot hard to read, and the table gives the numbers.

In [ ]:
# Colours of fkf.ipynb; the bank takes the violet of clipped-matched.ipynb (checked there for colour-blind
# readers against the other three: CVD Delta E >= 9.2).
COLOUR = {"minorized": "#2a78d6", "exact": "#eb6834", "matched": "#1baf7a", "bank": "#4a3aa7"}
STYLE = {"sKF": "-", "fKF": "--"}
t_change = (np.arange(2*N) - N)/FS


def plot_change(rows, title, t_max, ylim):
    keep = (t_change >= -0.25) & (t_change <= t_max)
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    for name, r in rows.items():
        family, method = name.split(", ")
        ax.plot(t_change[keep], 10*np.log10(r["curve"][keep]), color=COLOUR[method], ls=STYLE[family],
                lw=1.4, label=name)
    ax.axvline(0, color="k", ls="-.", lw=0.9)
    ax.axhline(TARGET_DB, color="0.45", lw=0.8)
    ax.text(t_max, TARGET_DB + 0.4, f"target, {TARGET_DB:.0f} dB", ha="right", va="bottom", fontsize=8, color="0.3")
    ax.set(xlabel="time from the change [s]", ylabel="misalignment [dB]", ylim=ylim, xlim=(-0.25, t_max),
           title=title)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc="upper right", ncol=2)
    plt.show()


plot_change(rows, r"Across $h_o \to -h_o$, AR(-0.9), $\beta^* = 0.2$, 5 dB, $b_\eta = \mathrm{E}|\eta|$",
            2.5, (-32, 8))

**Figure 2.** The posterior of the banks, $\mu_{t,j}$ averaged over the 20 realisations, over the whole change
run (left, 400-step blocks) and around the change (right, 20-step blocks). White, dashed: the change.
Colour map: cividis, made for colour-blind readers.

In [ ]:
def plot_posterior(banks, title):
    fig, axes = plt.subplots(len(banks), 2, figsize=(12, 2.9*len(banks)), constrained_layout=True,
                             gridspec_kw={"width_ratios": [2, 1]}, squeeze=False)
    for row, (label, grid, mu) in enumerate(banks):
        K = len(grid)
        for col, (lo, hi, block) in enumerate([(0, 2*N, 400), (N - FS//4, min(N + int(2.5*FS), 2*N), 20)]):
            ax = axes[row, col]
            blocks = (hi - lo)//block
            image = mu[:, lo:lo + blocks*block].reshape(K, blocks, block).mean(axis=2)
            shown = ax.imshow(image, aspect="auto", origin="lower", cmap="cividis", vmin=0, vmax=1,
                              interpolation="nearest",
                              extent=((lo - N)/FS, (lo + blocks*block - N)/FS, -0.5, K - 0.5))
            ax.axvline(0, color="w", ls="--", lw=1.0)
            ax.set_yticks(range(K), [f"{g:.1e}" for g in grid], fontsize=7)
            ax.set_title(f"{label}" + ("" if col == 0 else ", around the change"), fontsize=9)
            if col == 0:
                ax.set_ylabel(r"$\varepsilon^{(j)}$" if label.startswith("sKF") else r"$v^{(j)}$")
        axes[row, 0].set_xlabel("time from the change [s]")
        axes[row, 1].set_xlabel("time from the change [s]")
    fig.colorbar(shown, ax=axes, label=r"$\mu_{t,j}$, mean over realisations", shrink=0.8)
    fig.suptitle(title)
    plt.show()


plot_posterior([("sKF bank", EPS_BANK, rows["sKF, bank"]["mu"]), ("fKF bank", V_BANK, rows["fKF, bank"]["mu"])],
               "AR(-0.9)")
for family, grid in (("sKF", EPS_BANK), ("fKF", V_BANK)):
    mu = rows[f"{family}, bank"]["mu"]
    late = mu[:, N - N//4:N].mean(axis=1)
    after = mu[:, N:N + 2000].mean(axis=1)
    print(f"{family} bank, mean weights in the last quarter before the change: "
          + " ".join(f"{g:.1e}: {m:.3f}" for g, m in zip(grid, late)))
    print(f"{' '*len(family)}       in the first 2000 steps after it:            "
          + " ".join(f"{g:.1e}: {m:.3f}" for g, m in zip(grid, after)))

### 7.2 The exact filters at the banks' own floors

The banks do not land on $-20$ dB, so "own floor + 3 dB" compares different levels. Here the exact sKF and fKF
are tuned to each bank's floor at rest, with notebook 09's search (grids logspace$(-8, -3, 11)$ for $\varepsilon$,
logspace$(-6, -1, 11)$ for $v$, settled points only, interpolated), from `clipped-matched.ipynb` section 5 with the
filter as a parameter. The same search at $-20$ dB is the control: it must give back the tuned values above.
If a bank's floor lies outside what the search reaches with settled points, no filter is made at that floor and the
notebook says so.

In [ ]:
EPS_GRID = np.logspace(-8, -3, 11)          # notebook 09
V_GRID = np.logspace(-6, -1, 11)            # fkf.ipynb, leszek-setup.ipynb


def sweep(family, chi_fn, scale, grid, X, D):
    # notebook 09's search, as clipped-matched.ipynb section 5: every grid point at R realisations at once
    g = len(grid)
    Xg, Dg, values = np.repeat(X, g, axis=0), np.repeat(D, g, axis=0), np.tile(grid, len(X))
    mis = (sKF_batch(Xg, Dg, ho, scale, values, chi_fn) if family == "sKF"
           else fKF_batch(Xg, Dg, ho, scale, values, chi_fn))
    curves = mis.reshape(len(X), g, -1).mean(axis=0)
    del mis
    return (np.array([steady_state(c)[0] for c in curves]),
            np.array([is_settled(c, DRIFT_DB) for c in curves]))


def pick(grid, floors, settled, target):
    values, kept = grid[settled], floors[settled]
    if target < kept.min():
        return values[np.argmin(kept)], "target below every settled floor"
    if target > kept.max():
        return values[np.argmax(kept)], "target above every settled floor"
    order = np.argsort(kept)
    return 10**np.interp(target, kept[order], np.log10(values)[order]), "ok"


def equal_floor(family, sig, scale, bank_floor, tuned):
    grid = EPS_GRID if family == "sKF" else V_GRID
    floors, settled = sweep(family, chi_laplacian, scale, grid, *sig[0])
    control, _ = pick(grid, floors, settled, TARGET_DB)
    value, status = pick(grid, floors, settled, bank_floor)
    print(f"{family}: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in zip(floors, settled))
          + "   (* not settled)")
    print(f"   at -20 dB: {control:.4e} (tuned value {tuned:.4e}, ratio {control/tuned:.5f})")
    print(f"   at the bank's floor, {bank_floor:.2f} dB: {value:.4e}, {status}")
    if status != "ok":
        print("   no filter at the bank's floor: the search does not reach it with settled points")
        return None
    return single_row(family, chi_laplacian, scale, value, sig)


start = time.time()
rows_equal = {}
for family in ("sKF", "fKF"):
    rows_equal[f"{family}, exact, -20 dB"] = rows[f"{family}, exact"]
    row = equal_floor(family, SIG, B_ETA, rows[f"{family}, bank"]["floor"], TUNED[(family, "exact")])
    if row is not None:
        rows_equal[f"{family}, exact, bank floor"] = row
    rows_equal[f"{family}, bank"] = rows[f"{family}, bank"]
print(f"[{time.time() - start:.0f} s]\n")
show_table(rows_equal)

### 7.3 The Gaussian likelihood

One table of its own, not in the comparison above: the sKF bank with the Gaussian likelihood
($v_\eta$ = the true noise variance), against the Gaussian sKF (the Kalman filter's scalar form, eq. chi.Gaussian)
tuned to $-20$ dB with the same search. If no settled grid point reaches $-20$ dB, the search returns the deepest
settled point, as notebook 09's protocol does, and says so.

In [ ]:
start = time.time()
LIK_G = likelihood("Gaussian", -0.9)
floors_g, settled_g = sweep("sKF", chi_gaussian, LIK_G[2], EPS_GRID, *SIG[0])
value_g, status_g = pick(EPS_GRID, floors_g, settled_g, TARGET_DB)
print("Gaussian sKF, grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in zip(floors_g, settled_g))
      + "   (* not settled)")
print(f"   at -20 dB: eps = {value_g:.4e}, {status_g}")
rows_g = {"sKF, Gaussian, tuned": single_row("sKF", chi_gaussian, LIK_G[2], value_g, SIG),
          "sKF, Gaussian bank": bank_row("sKF", LIK_G, SIG)}
print(f"[{time.time() - start:.0f} s]\n")
show_table(rows_g)

## 8. White input

The same code with the AR coefficient at 0 (signal power 1, so the noise and $b_\eta$ follow). The draft says
the gain of the bank shows with white input. Banks against the exact filters at the white-input values of
`leszek-setup.ipynb` section 5 ($\varepsilon = 1.319 \times 10^{-6}$, $v = 3.017 \times 10^{-4}$, printed with
four digits), and against the exact filters tuned to the banks' floors.

In [ ]:
start = time.time()
SIG_W = scenario_signals(0.0)
_, B_ETA_W = noise_of(0.0)
LIK_W = likelihood("Laplacian", 0.0)
TUNED_W = {"sKF": 1.319e-6, "fKF": 3.017e-4}
PRINTED_W = {"sKF": (-20.01, 1564, 1221), "fKF": (-20.00, 1054, 1596)}
rows_w = {}
for family in ("sKF", "fKF"):
    rows_w[f"{family}, exact"] = single_row(family, chi_laplacian, B_ETA_W, TUNED_W[family], SIG_W)
    rows_w[f"{family}, bank"] = bank_row(family, LIK_W, SIG_W)
print("control: the exact filters against the numbers printed by leszek-setup.ipynb section 5")
for family, (floor, steps, rec) in PRINTED_W.items():
    r = rows_w[f"{family}, exact"]
    print(f"   {family}, exact  floor {r['floor']:.2f} / {floor:.2f}, steps {r['reached']} / {steps}, "
          f"recovery {r['rec']} / {rec}")
rows_w_equal = {}
for family in ("sKF", "fKF"):
    rows_w_equal[f"{family}, exact, -20 dB"] = rows_w[f"{family}, exact"]
    row = equal_floor(family, SIG_W, B_ETA_W, rows_w[f"{family}, bank"]["floor"], TUNED_W[family])
    if row is not None:
        rows_w_equal[f"{family}, exact, bank floor"] = row
    rows_w_equal[f"{family}, bank"] = rows_w[f"{family}, bank"]
print(f"[{time.time() - start:.0f} s]\n")
show_table(rows_w_equal)
print(f"largest |u| met by the banks: sKF {rows_w['sKF, bank']['u_max']:.1e}, fKF {rows_w['fKF, bank']['u_max']:.1e}")

**Figure 3.** White input: the exact filters at $-20$ dB and the banks across the change (same encoding as
Figure 1), and the posterior of the banks (as Figure 2).

In [ ]:
plot_change(rows_w, r"Across $h_o \to -h_o$, white input, $\beta^* = 0.2$, 5 dB, $b_\eta = \mathrm{E}|\eta|$",
            0.6, (-32, 8))
plot_posterior([("sKF bank", EPS_BANK, rows_w["sKF, bank"]["mu"]), ("fKF bank", V_BANK, rows_w["fKF, bank"]["mu"])],
               "white input")

### 8.1 The short spikes in the bank curves

The mean curves of the banks have short upward spikes (Figures 1 and 3). The largest one, in the white-input fKF bank
0.24 s before the change, is realisation 11 at step 94 072. That realisation is run again up to there with the
weights recorded. Columns: the noise at step $t$ in units of $b_\eta$; the weights after step $t$ of the smallest and
the largest $v$; the evidence of step $t$ for $v = 10^{-2}$ against $v = 10^{-6}$, $\log p_{t,9}(e_{t,9}) - \log p_{t,1}(e_{t,1})$;
and the misalignment of the output after step $t$.

In [ ]:
r_spike, t_spike = 11, 94072
x_s, d_s = signals(r_spike, 0.0, 2*N, change_at=N)
out_s = bank_batch(x_s[None, :t_spike + 4], d_s[None, :t_spike + 4], ho, V_BANK, Q_BANK, "fKF", *LIK_W, flip_at=N,
                   record=True)
rng_s = np.random.default_rng(r_spike)                 # the noise of that realisation, drawn as signals() draws it
rng_s.standard_normal(2*N + WARMUP)
eta_s = gennorm.rvs(BETA, scale=noise_of(0.0)[0], size=2*N, random_state=rng_s)
print(f"{'step':>7}{'eta/b_eta':>11}{'mu, v = 1e-6':>14}{'mu, v = 1e-2':>14}{'evidence':>10}{'output after [dB]':>19}")
for t in range(t_spike - 4, t_spike + 3):
    mu_t = np.exp(out_s["log_mu"][:, 0, t])
    evidence = out_s["log_p"][-1, 0, t] - out_s["log_p"][0, 0, t]
    print(f"{t:>7d}{eta_s[t]/B_ETA_W:>11.1f}{mu_t[0]:>14.3e}{mu_t[-1]:>14.3e}{evidence:>10.1f}"
          f"{10*np.log10(out_s['mis'][0, t + 1]):>19.2f}")
print("\nlimit of the evidence of one large error, eq. (bounded.evidence), (sigma_j^2 - sigma_1^2)/(2 b_eta^2), "
      "||x||^2 = M:")
print("   fKF bank, v = " + ", ".join(f"{v:.0e}: {(v - V_BANK[0])*M/(2*B_ETA_W**2):.2f}" for v in V_BANK[[4, 6, 8]]))
print(f"   sKF bank, eps = 1e-4 (at rest v_bar << eps): {(EPS_BANK[-1] - EPS_BANK[0])*M/(2*B_ETA_W**2):.2f}")

## 9. Findings

**Checks: all pass, with both likelihoods.**

1. $K = 1$: the bank gives the same numbers as the single sKF and fKF, bit for bit, at $q = 0$ and $q = 10^{-4}$.
2. $q = 0$: every filter of the bank gives the same numbers, bit for bit, as the single filter at its grid value.
   The weights equal eq. (eps.likelihood) built from the single filters' own errors, to $|\Delta\mu| \le 10^{-11}$.
3. The rows of $p_{ij}$ sum to 1 exactly. In runs, $\sum_j c_{t,j}$, $\sum_j \mu_{t,j}$ and $\sum_i \mu_{t-1,i\mid j}$
   are 1 to $10^{-13}$. With outliers up to $10^9$ added, everything stays finite. The Laplacian sums then
   drift by $4 \times 10^{-7}$, because of the density formula (point 2 of the notes below).
4. Brute force: the bank matches the exact mixture over $\varepsilon$ sequences at the first update ($\le 2 \times 10^{-16}$)
   and, for $q = 0$, at every update. After the first update the gap grows with $q$: the largest $|\Delta\mu|$ over 12
   updates is $2 \times 10^{-6}$ to $8 \times 10^{-5}$ at $q = 10^{-4}$ and up to $3 \times 10^{-2}$ at $q = 0.2$.
   That gap is the cost of the merge.
5. The Laplacian log density matches direct quadrature to $3 \times 10^{-8}$ and integrates to 1.

**Anchor.** Leszek's numbers are reproduced on his signals:
- best fixed $\varepsilon = 10^{-5}$: $-8.98$ dB
- bank at $q = 10^{-4}$: $-10.75$ dB
- bank at $q = 0$ with his clamp: $-9.87$ dB
- bank at $q = 10^{-3}$: $-10.76$ dB
- Gaussian likelihood: best fixed $-7.65$ dB and bank $-8.52$ dB (the draft's table has $-7.7$ and $-8.5$).

**Our scenario, AR($-0.9$): the banks do no better than the tuned single filters.**

| | sKF bank | single sKFs | fKF bank | single fKFs |
|---|---|---|---|---|
| floor at rest [dB] | $-25.9$ (not settled) | $-20$ | $-15.7$ (not settled) | $-20$ |
| at rest, steps to $-17$ dB | 36 289 | 3 631 - 4 751 | never | 4 188 - 5 462 |
| after the flip, 0 dB | **924** | 1 606 - 1 747 | **886** | 2 022 - 2 959 |
| $-5$ dB | **1 736** | 2 954 - 3 312 | **1 702** | 3 699 - 4 387 |
| $-10$ dB | 5 832 | 4 094 - 4 834 | 7 494 | 5 308 - 5 565 |
| $-15$ dB | 31 119 | 5 345 - 6 497 | 72 194 | 6 840 - 7 154 |
| back below $-20$ dB | 59 834 | 8 793 - 12 224 | never | 11 776 - 12 277 |
| mean over the record [dB] | $-13.89$ | $-13.94$ to $-14.08$ | $-11.92$ | $-12.41$ to $-13.45$ |

- **The posterior moves at the change.** At rest it sits on the smallest grid value ($\varepsilon = 10^{-9}$ and
  $v = 10^{-6}$, weight 0.987 each). The system is static, so that is the right value. At the flip the weight moves
  at once to the largest values: in the first 2000 steps, the sKF bank puts 0.95 on $10^{-6}$ to $10^{-4}$. The
  banks then start faster than every single filter, down to $-5$ dB.
- **The posterior then comes back too early.** It returns to the smallest values within about 0.5 s (Figure 2), when
  the misalignment is still near $-8$ dB, and the banks then crawl. The same happens in the initial convergence
  from zero. This is what the draft says for coloured input (line 1558): "the bank then performs only as well as
  the best fixed $\varepsilon$". A likely reason, as the draft explains, is that the prediction errors carry little of
  the misalignment that lies where the AR($-0.9$) input has little energy. This notebook does not test that reason.
- **Mean over the record.** The sKF bank is within 0.2 dB of the single sKFs. The fKF bank is 0.5 to 1.5 dB worse
  than the single fKFs.
- The banks' floors are not at $-20$ dB, so their "own floor + 3 dB" column compares different levels. Section 7.2
  compares them at equal floors:
  - **sKF.** The exact sKF at the bank's floor ($\varepsilon = 2.84 \times 10^{-7}$, $-25.93$ dB) has a mean of
    $-12.37$ dB, so the bank is 1.5 dB better on the mean. The bank is faster down to $-10$ dB after the flip
    (924 / 1 736 / 5 832 steps against 3 357 / 6 369 / 9 233). It is slower below that: $-15$ dB at 31 119
    against 11 989.
  - **fKF.** The exact fKF at the bank's floor ($v = 7.25 \times 10^{-4}$, $-15.60$ dB) has a mean of $-13.13$ dB,
    1.2 dB better than the bank.
- **Controls.** The six single filters reproduce `leszek-setup.ipynb` section 4 exactly (floor, steps, recovery).
  The search at $-20$ dB gives back the tuned exact values (ratio 0.99997 and 1.00000).

**Gaussian likelihood, AR($-0.9$).**
- The Gaussian sKF cannot reach $-20$ dB with a settled grid point. Its deepest settled point is $-19.70$ dB at
  $\varepsilon = 10^{-7}$, and it needs 32 148 steps to come within 3 dB of it.
- The Gaussian sKF bank settles at $-10.0$ dB. Its mean over the record is $-8.41$ dB, against $-7.42$ dB for the
  single filter.

**White input: the banks gain clearly.**

| | sKF bank | exact sKF | fKF bank | exact fKF |
|---|---|---|---|---|
| floor at rest [dB] | $-44.9$ | $-20.0$ | $-44.6$ | $-20.0$ |
| at rest, steps to $-17$ / $-20$ dB | **1 473 / 1 670** | 1 562 / 2 823 | **1 047 / 1 317** | 1 056 / 1 630 |
| after the flip, 0 / $-10$ / $-20$ dB | **177 / 616 / 1 106** | 466 / 899 / 1 742 | **175 / 737 / 1 304** | 676 / 1 260 / 2 113 |
| mean over the record [dB] | **$-22.64$** | $-17.22$ | **$-23.01$** | $-16.67$ |

- The banks have a floor 25 dB deeper, are faster at every level, and are 5.4 / 6.3 dB better on the mean.
- There is no equal-floor reference: no settled point of notebook 09's grid reaches $-45$ dB within $N$ steps.
- The fKF's steps at rest are 1 056 against the printed 1 054. The value given to four digits ($3.017 \times 10^{-4}$)
  differs from the search's $3.0175 \times 10^{-4}$.

**The spikes (section 8.1).**
- A Laplacian error moves the weights by a bounded amount (eq. bounded.evidence), but the bound
  $(\sigma_j^2 - \sigma_i^2)/(2b_\eta^2)$ is about 32 nats for $v = 10^{-2}$ against $v = 10^{-6}$, because then
  $\sigma_j \approx 8\,b_\eta$.
- Two outliers in a row (40 and 150 $b_\eta$) moved the weight of $v = 10^{-2}$ from about $10^{-17}$ to 0.95. That
  filter had just made its largest correction, so the output's misalignment went to about 0 dB, and it came back
  within 4 steps.
- The sKF bank's largest value, $\varepsilon = 10^{-4}$, has a bound of about 0.3 nats, so its spikes are small.
- Only the largest spike was examined.

**Differences between the draft and Leszek's scripts.**
1. His bank loops in `driftbench.py`, `bankbench.py`, `driftcmp.py` and `fkfbank*.py` follow Algorithms 1 and 2.
2. **The weight clamp.** He clamps $\mu \ge 10^{-280}$ (`driftbench*.py`) or $10^{-200}$ (`driftcmp.py`,
   `fkfbank*.py`); the draft has no clamp. At $q = 10^{-4}$ it changes nothing ($-10.75$ dB with and without). At
   $q = 0$ it decides the result: $-9.87$ dB with the clamp, **$-7.21$ dB without**. Without the clamp, the bank
   with no mixing loses all of the gain over the best fixed $\varepsilon$ ($-8.98$ dB) and 1.8 dB more, not "about
   half of the gain" (line 1476).
3. **The `bankbench.py` header comment.** It says the output "averages correction and variance over the eps
   posterior". The code outputs $\hat{\boldsymbol w}_t = \sum_j \mu_{t,j}\boldsymbol w_{t,j}$, as eq.
   (bank.output) says. The comment describes `mixlap.skf_mix`, a different model.
4. His `Pmat` gives $p_{11} = 1 - q$ for $K = 1$, where eq. (eps.chain) gives 1. This only matters for $K = 1$.

**Notes on the draft.**
1. Algorithms 1 and 2 are implemented as written. The only additions:
   - the weights are carried as $\log\mu$, the same numbers;
   - the bank does nothing before the window is full.
2. Eq. (bank.weights) as written adds $\log\phi(u) \approx -u^2/2$ and $\log\mathcal R(k - u) \approx +u^2/2$, which
   cancel. The error is $6 \times 10^{-7}$ at $|u| = 10^5$ and $5 \times 10^{-3}$ at $|u| = 10^7$, and the result
   is wrong beyond $10^8$. Our runs met $|u| \le 8.5 \times 10^3$, where the error is below $10^{-8}$. A form
   without the cancellation is
   $k^2/2 + \log\big(e^{-ku}\Phi(u - k) + e^{ku}\Phi(-u - k)\big) - \log 2b_\eta$.

**Not done.**
- Items 2-6 of the draft's open-task list.
- Sensitivity to $q$, $K$ or the grid range.
- The minorized and matched single filters with white input; only the exact ones were run there.

In [ ]:
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min")